# M2 - Khám phá tập Train của ViHOS

Notebook này chỉ giữ lại các phần cần thiết cho báo cáo đồ án:

- Đọc 3 phiên bản train: Syllable, Word và Span Extraction.
- Kiểm tra các phiên bản có khớp số comment hay không.
- Xem một comment thật ở cả 3 dạng.
- Thống kê số comment có span / comment sạch.
- Thống kê số span trên mỗi comment.
- Thống kê độ dài comment ở mức Syllable và Word.
- Thống kê phân bố nhãn BIO.

> `sentence_id` là khóa dùng để nhóm các đơn vị thuộc cùng một comment.


In [57]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "DOCS" else current_dir
data_root = project_root / "DATA" / "ViHOS_official"

syllable_train_path = (
    data_root / "train_sequence_labeling" / "syllable" / "train_BIO_syllable.csv"
)
word_train_path = (
    data_root / "train_sequence_labeling" / "word" / "train_BIO_Word.csv"
)
span_train_path = data_root / "train_span_extraction" / "train.csv"

df_syllable = pd.read_csv(syllable_train_path)
df_word = pd.read_csv(word_train_path)
df_span = pd.read_csv(span_train_path)

print("Đã đọc xong 3 file train.")
print("Syllable:", df_syllable.shape)
print("Word:", df_word.shape)
print("Span Extraction:", df_span.shape)


Đã đọc xong 3 file train.
Syllable: (112754, 5)
Word: (108432, 5)
Span Extraction: (8844, 3)


In [58]:
# Kiểm tra các phiên bản train có cùng số comment

n_syllable = df_syllable["sentence_id"].nunique()
n_word = df_word["sentence_id"].nunique()
n_span = len(df_span)

print("Số comment - Syllable:", n_syllable)
print("Số comment - Word:", n_word)
print("Số comment - Span Extraction:", n_span)

assert n_syllable == n_word == n_span, "Ba phiên bản train không khớp số comment."

print("\nPASS: Ba phiên bản train khớp nhau về số comment.")


Số comment - Syllable: 8844
Số comment - Word: 8844
Số comment - Span Extraction: 8844

PASS: Ba phiên bản train khớp nhau về số comment.


In [59]:
# Xem cùng một comment thật ở cả 3 phiên bản

sentence_id = 1

sample_syllable = (
    df_syllable[df_syllable["sentence_id"] == sentence_id]
    [["Word", "Tag"]]
    .reset_index(drop=True)
)

sample_word = (
    df_word[df_word["sentence_id"] == sentence_id]
    [["Word", "Tag"]]
    .reset_index(drop=True)
)

sample_original = df_span.iloc[sentence_id]

print("CÂU GỐC:")
print(sample_original["content"])

print("\nSYLLABLE:")
display(sample_syllable)

print("\nWORD:")
display(sample_word)

print("\nINDEX SPANS:")
print(sample_original["index_spans"])


CÂU GỐC:
Bấp bênh vl thế

SYLLABLE:


,Word,Tag
0,Bấp,O
1,bênh,O
2,vl,B-T
3,thế,O



WORD:


,Word,Tag
0,Bấp_bênh,O
1,vl,B-T
2,thế,O



INDEX SPANS:
[9, 10]


In [60]:
# Đếm số span của từng comment.
# Mỗi B-T đánh dấu sự bắt đầu của một span mới.

span_counts = (
    df_syllable.assign(is_span_start=df_syllable["Tag"].eq("B-T"))
    .groupby("sentence_id")["is_span_start"]
    .sum()
)

num_comments = df_syllable["sentence_id"].nunique()
num_with_span = int((span_counts > 0).sum())
num_clean = int((span_counts == 0).sum())
total_spans = int(span_counts.sum())
avg_spans_positive = span_counts[span_counts > 0].mean()
max_spans = int(span_counts.max())

print("Tổng số comment:", num_comments)
print("Comment có ít nhất 1 span:", num_with_span)
print("Comment sạch:", num_clean)
print("Tổng số span:", total_spans)
print("Trung bình span / comment có span:", round(avg_spans_positive, 2))
print("Số span nhiều nhất trong một comment:", max_spans)


Tổng số comment: 8844
Comment có ít nhất 1 span: 4292
Comment sạch: 4552
Tổng số span: 10075
Trung bình span / comment có span: 2.35
Số span nhiều nhất trong một comment: 23


In [61]:
# Phân bố số span trên mỗi comment

span_distribution = (
    span_counts
    .value_counts()
    .sort_index()
    .rename_axis("Số span trong comment")
    .reset_index(name="Số comment")
)

span_distribution


,Số span trong comment,Số comment
0,0,4552
1,1,1972
2,2,992
3,3,535
4,4,310
5,5,191
6,6,98
7,7,64
8,8,47
9,9,31


In [62]:
# Độ dài comment ở mức Syllable và Word

lengths_syllable = df_syllable.groupby("sentence_id").size()
lengths_word = df_word.groupby("sentence_id").size()

length_summary = pd.DataFrame({
    "Biểu diễn": ["Syllable", "Word"],
    "Trung bình": [lengths_syllable.mean(), lengths_word.mean()],
    "Trung vị": [lengths_syllable.median(), lengths_word.median()],
    "Ngắn nhất": [lengths_syllable.min(), lengths_word.min()],
    "Dài nhất": [lengths_syllable.max(), lengths_word.max()],
}).round(2)

length_summary


,Biểu diễn,Trung bình,Trung vị,Ngắn nhất,Dài nhất
0,Syllable,12.75,9.0,1,162
1,Word,12.26,8.0,1,168


In [63]:
# Phân bố nhãn BIO ở mức Syllable

tag_counts = df_syllable["Tag"].value_counts()
tag_percent = (df_syllable["Tag"].value_counts(normalize=True) * 100).round(2)

tag_summary = pd.DataFrame({
    "Số lượng": tag_counts,
    "Tỷ lệ (%)": tag_percent,
})

tag_summary


,Số lượng,Tỷ lệ (%)
Tag,,
O,91598,81.24
I-T,11081,9.83
B-T,10075,8.94


In [64]:
# Bảng tóm tắt train dùng trực tiếp khi viết phần ngữ liệu trong báo cáo

train_summary = pd.DataFrame({
    "Chỉ số": [
        "Số comment train",
        "Comment có span",
        "Comment sạch",
        "Tổng số span",
        "Tỷ lệ comment có span (%)",
        "Tỷ lệ comment sạch (%)",
        "Span trung bình / comment có span",
        "Số span tối đa / comment",
        "Độ dài Syllable trung bình",
        "Độ dài Syllable trung vị",
        "Độ dài Syllable lớn nhất",
        "Độ dài Word trung bình",
        "Độ dài Word trung vị",
        "Độ dài Word lớn nhất",
    ],
    "Giá trị": [
        num_comments,
        num_with_span,
        num_clean,
        total_spans,
        round(num_with_span / num_comments * 100, 2),
        round(num_clean / num_comments * 100, 2),
        round(avg_spans_positive, 2),
        max_spans,
        round(lengths_syllable.mean(), 2),
        round(lengths_syllable.median(), 2),
        int(lengths_syllable.max()),
        round(lengths_word.mean(), 2),
        round(lengths_word.median(), 2),
        int(lengths_word.max()),
    ],
})

train_summary


,Chỉ số,Giá trị
0,Số comment train,8844.00
1,Comment có span,4292.00
2,Comment sạch,4552.00
3,Tổng số span,10075.00
4,Tỷ lệ comment có span (%),48.53
5,Tỷ lệ comment sạch (%),51.47
6,Span trung bình / comment có span,2.35
7,Số span tối đa / comment,23.00
8,Độ dài Syllable trung bình,12.75
9,Độ dài Syllable trung vị,9.00


## Những điều cần ghi nhớ từ M2

- `sentence_id` dùng để nhóm các đơn vị thuộc cùng một comment.
- Bản Syllable và Word là hai cách biểu diễn của cùng comment.
- `B-T` bắt đầu một span mới; `I-T` tiếp tục span hiện tại; `O` nằm ngoài span.
- Số `B-T` trong một comment = số span của comment đó.
- Comment không có `B-T` được xem là comment sạch.
- `O` thường chiếm phần lớn dữ liệu, vì vậy Accuracy không nên là metric chính cho bài toán này.
- Các bảng thống kê ở trên sẽ được dùng lại khi viết phần **Ngữ liệu** trong báo cáo cuối kỳ.
